# HISCALE Observations builder demo
This notebook loads the bundled HISCALE BEASD + AIMMS + miniSPLAT datasets, constructs a particle population via the `hiscale_observations` factory, then walks through the metadata, diagnostics, and a `frac_ccn`/state-line plot using the `state_line` viz factory.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from part2pop import build_population
from part2pop.analysis import build_variable
from part2pop.population.factory import hiscale_observations as hiscale
from part2pop.viz.factory.state_line import build as build_state_line


In [ ]:
data_dir = Path('examples/example_data/HISCALE_data_0425')
config = {
    'type': 'hiscale_observations',
    'aimms_file': str(data_dir / 'AIMMS20_G1_20160425155810_R2_HISCALE020h.txt'),
    'beasd_file': str(data_dir / 'BEASD_G1_20160425155810_R2_HISCALE_001s.txt'),
    'ams_file': str(data_dir / 'HiScaleAMS_G1_20160425_R0.txt'),
    'splat_file': str(data_dir / 'Splat_Composition_25-Apr-2016.txt'),
    'z': 100.0,
    'dz': 50.0,
    'mass_thresholds': {
        'BC': ((0.0, 0.5, 0.1), ['BC']),
        'OIN': ((0.0, 0.5, 0.1), ['OIN']),
    },
    'splat_species': {'BC': ['soot'], 'OIN': ['Dust']},
    'splat_number_fraction': {'BC': 0.65, 'OIN': 0.35},
}
pop = build_population(config)

In [ ]:
print('Created population with', pop.num_particles, 'particles')
print('Metadata keys:', pop.metadata.keys())
print('Size dist bins (nm):', np.array(pop.metadata['size_distribution']['Dp_hi_nm']) - np.array(pop.metadata['size_distribution']['Dp_lo_nm']))


## Diagnostics
Inspect the metadata for AMS mass fractions and miniSPLAT comparisons so we can verify the builder recorded the matching diagnostics correctly.

In [ ]:
print('AMS mass fractions:', pop.metadata['ams_mass_frac'])
print('miniSPLAT mass fractions:', pop.metadata['splat_mass_frac'])
print('miniSPLAT number fractions:', pop.metadata['splat_number_frac'])


## Comparison to BEASD observations
Plot the reconstructed size distribution metadata against the observed BEASD averages used to initialize the builder.

In [ ]:
obs_lo, obs_hi, obs_N_cm3, _ = hiscale._read_beasd_avg_size_dist(
    beasd_file=config['beasd_file'],
    aimms_file=config['aimms_file'],
    z=config['z'],
    dz=config['dz'],
    beasd_density_measure='per_bin',
)
obs_mid = obs_lo + 0.5 * (obs_hi - obs_lo)
pop_sd = pop.metadata['size_distribution']
pop_mid = pop_sd['Dp_lo_nm'] + 0.5 * (pop_sd['Dp_hi_nm'] - pop_sd['Dp_lo_nm'])
pop_N_cm3 = pop_sd['N_bin_m3'] * 1e-6
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pop_mid, pop_N_cm3, label='reconstructed', linewidth=2)
ax.plot(obs_mid, obs_N_cm3, label='BEASD observation', linestyle='--')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Dp (nm)')
ax.set_ylabel('N (cm^-3)')
ax.set_title('Reconstructed vs. observed BEASD size distribution')
ax.legend()


## miniSPLAT reduced-class comparison
Compare the builder-assigned reduced-class fractions to the averages from the original miniSPLAT table.

In [ ]:
splat_table = hiscale._read_delimited_table_with_header(config['splat_file'])
obs_reduced = {}
for reduced, cols in config['splat_species'].items():
    values = np.zeros_like(next(iter(splat_table.values())))
    for col in cols:
        values += np.asarray(splat_table[col], dtype=float)
    obs_reduced[reduced] = float(np.nanmean(values))
norm = sum(obs_reduced.values())
obs_reduced = {k: v / norm for k, v in obs_reduced.items()}
model_reduced = pop.metadata['type_fracs']
labels = sorted(obs_reduced.keys())
obs_vals = [obs_reduced[label] for label in labels]
model_vals = [model_reduced.get(label, 0.0) for label in labels]
x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width / 2, obs_vals, width, label='miniSPLAT obs')
ax.bar(x + width / 2, model_vals, width, label='Reconstruction')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('fraction')
ax.set_title('miniSPLAT reduced-class fractions')
ax.legend()


## frac_ccn state line
Create a `state_line` plotter for the `frac_ccn` variable, using the computed population to visualize how the derived fraction varies across the `s_grid`.

In [ ]:
state_line_cfg = {
    'type': 'state_line',
    'varname': 'frac_ccn',
    'var_cfg': {'s_grid': np.logspace(-2, 1, 50)},
    'style': {'marker': 'o', 'linewidth': 2},
}
plotter = build_state_line(state_line_cfg)
fig, ax = plt.subplots(figsize=(6, 4))
plotter.plot(pop, ax, add_xlabel=True, add_ylabel=True)
ax.set_title('frac_ccn state line from HISCALE builder')
fig.tight_layout()


## Additional diagnostics
Compute summary metrics (total N, total mass, matching diagnostics) and compare the builder metadata to the observed FIMS/BEASD totals so readers can reconcile why CCN fractions behave the way they do.

In [ ]:
total_obs = np.sum(obs_N_cm3)
total_recon = np.sum(pop_N_cm3)
print(f'BEASD total N (cm^-3): {total_obs:.3f}')
print(f'Reconstructed total N (cm^-3): {total_recon:.3f}')
if 'ams_mass_frac' in pop.metadata:
    print('AMS mass fractions (metadata):', pop.metadata['ams_mass_frac'])
ratio = total_recon / total_obs if total_obs > 0 else np.nan
print(f'Reconstructed/observed total N ratio: {ratio:.2f}')
